# Training with `Trainer`

A tour of `lwfm.training` on a small synthetic task. Everything runs on CPU in a few
seconds, so the numbers show the code working, not a good model.

1. The simplest run
2. Validation, logging and a learning-rate schedule
3. Callbacks: react to training without editing the loop
4. Distillation: a frozen teacher is just another node
5. Saving models through the sinks
6. Mistakes fail before training starts
7. Loop settings, and many runs with one trainer

Run from `examples/`. `lwfm` isn't an installed package yet, so put the repository root on the import path.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

## The data

Four classes. Each class has one fixed random "prototype" image, and every sample is a
noisy copy of its class's prototype, so a small linear model can learn the task.

Batches use lwfm's layout, `{"inputs": {...}, "targets": {...}}`. The trainer accepts
any iterable of such dicts as a loader: a plain list here, a `DataLoader` in real runs.

In [2]:
import torch

K = 4  # number of classes
# One fixed random 3x8x8 image per class
prototypes = torch.rand(K, 3, 8, 8, generator=torch.Generator().manual_seed(0))


def make_batches(count: int, batch_size: int = 32, seed: int = 0) -> list[dict]:
    """`count` batches of `batch_size` noisy prototype images with their class labels,
    in lwfm's batch layout. The same seed always gives the same batches."""
    generator = torch.Generator().manual_seed(seed)
    batches = []
    for _ in range(count):
        labels = torch.randint(0, K, (batch_size,), generator=generator)
        noise = torch.randn(batch_size, 3, 8, 8, generator=generator)
        # A sample is its class's prototype plus noise
        images = prototypes[labels] + 1.5 * noise
        batches.append({"inputs": {"image": images}, "targets": {"class": labels}})
    return batches


train_loader = make_batches(20)  # 20 batches make one epoch
val_loader = make_batches(5, seed=1)  # new samples from the same classes
train_loader[0]["inputs"]["image"].shape, train_loader[0]["targets"]["class"][:8]

(torch.Size([32, 3, 8, 8]), tensor([0, 3, 1, 0, 3, 3, 3, 3]))

## 1. The simplest run

A model becomes a one-node graph, and the loss reads its inputs by name. The trainer
takes any torch optimizer, already built: it never chooses one for you.

`Trainer(...)` holds the loop settings (device, how long to train, ...). `fit(...)`
takes the objects for one run (graph, data, loss, optimizer).

In [3]:
from torch import nn

from lwfm.core.graph import ExecutionGraph, ModelNode
from lwfm.core.losses import LossBinding, LossComposer
from lwfm.training import Trainer, seed_everything


def make_student() -> ModelNode:
    """A fresh one-node student: a linear classifier that reads `inputs.image` and
    publishes `student.logits`."""
    # Any nn.Module works here, unchanged
    return ModelNode(
        "student",
        nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, K)),
        inputs="inputs.image",
        outputs="logits",
    )


seed_everything(0)  # before building the model: weight init draws from the seed
student = make_student()
graph = ExecutionGraph([student])
loss = LossComposer(
    [LossBinding("ce", nn.CrossEntropyLoss(), ["student.logits", "targets.class"])]
)
optimizer = torch.optim.SGD(student.parameters(), lr=0.1)

result = Trainer(device="cpu", max_epochs=3).fit(graph, train_loader, loss, optimizer)
# 3 epochs x 20 batches = 60 optimizer steps; no validation ran, so history is empty
result

TrainingResult(global_step=60, epochs_completed=3, stop_reason='max_epochs', history=[])

## 2. Validation, logging and a learning-rate schedule

- **Metrics** are torchmetrics objects bound to names, just like losses. Given
  `val_loader` and `metrics`, the trainer validates after every epoch.
- **Sinks** receive every result as an event. `ConsoleSink` prints them; with no sink,
  nothing is written anywhere.
- **Schedulers** are plain torch schedulers, and need `scheduler_interval`: `"step"`
  (after each optimizer step) or `"epoch"`. There is deliberately no default, since
  the two differ by a factor of the epoch length.

In [4]:
from torchmetrics.classification import MulticlassAccuracy

from lwfm.evaluation.metrics import MetricBinding
from lwfm.results.sinks import ConsoleSink

metrics = [
    MetricBinding(
        "top1",
        MulticlassAccuracy(K, average="micro"),
        ["student.logits", "targets.class"],
    )
]

seed_everything(0)
student = make_student()
graph = ExecutionGraph([student])
optimizer = torch.optim.SGD(student.parameters(), lr=0.1)
epochs = 3
# Cosine decay to 0 over the whole run, so T_max counts optimizer steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs * len(train_loader)
)

trainer = Trainer(device="cpu", max_epochs=epochs, log_every_n_steps=20)
result = trainer.fit(
    graph,
    train_loader,
    loss,
    optimizer,
    val_loader=val_loader,
    metrics=metrics,
    scheduler=scheduler,
    scheduler_interval="step",
    output_sink=ConsoleSink(),
)
result.best("top1")  # the best validation, and the step it ran at

[step 20] train/loss/ce = 0.4182


[step 20] train/loss/total = 0.4182


[step 20] train/lr = 0.075


[step 20] train/epoch_seconds = 0.003208


[step 20] eval/val/top1 = 0.9


[step 40] train/loss/ce = 0.2435


... (6 outputs left out)

[step 60] train/lr = 0


[step 60] train/epoch_seconds = 0.0032


[step 60] eval/val/top1 = 0.8875


{'top1': 0.8999999761581421, 'step': 20}

## 3. Callbacks: react to training without editing the loop

A callback is an object whose `on_*` methods the trainer calls at fixed points, such as
`on_step_end`, `on_validation_end` and `on_fit_end`. Each hook receives the
`TrainingState`: the graph, the optimizer, the step counters and the latest metrics.
Setting `state.should_stop = True` ends the run cleanly.

`lwfm.training.callbacks` has ready-made ones: `EarlyStopping`, `ModelCheckpoint`,
`LossWeightSchedule` and `EMAUpdate`.

### When each hook runs

Each box is a loop or a step that only sometimes happens; the hooks inside it run top
to bottom. A box nested inside another runs at that point in its parent's list.

```
+----------------------------------------------------------------------------------+
| FIT   one call to trainer.fit(...)                                               |
|   - on_fit_start                                                                 |
|   +----------------------------------------------------------------------------+ |
|   | EPOCH   repeats until max_epochs, max_steps or state.should_stop           | |
|   |   - on_epoch_start                                                         | |
|   |   +----------------------------------------------------------------------+ | |
|   |   | BATCH   repeats for each batch of train_loader                       | | |
|   |   |   - on_step_start(batch)                                             | | |
|   |   |   - on_after_backward                                                | | |
|   |   |   +----------------------------------------------------------------+ | | |
|   |   |   | OPTIMIZER STEP   only at an accumulation boundary              | | | |
|   |   |   |   - on_after_optimizer_step                                    | | | |
|   |   |   +----------------------------------------------------------------+ | | |
|   |   |   - on_step_end(output)                                              | | |
|   |   |   +----------------------------------------------------------------+ | | |
|   |   |   | VALIDATION   at a boundary, every val_every_n_steps (if set)   | | | |
|   |   |   |   - on_validation_end(metrics)                                 | | | |
|   |   |   +----------------------------------------------------------------+ | | |
|   |   +----------------------------------------------------------------------+ | |
|   |   - on_epoch_end            (skipped if training stopped mid-epoch)        | |
|   |   +----------------------------------------------------------------------+ | |
|   |   | VALIDATION   after the epoch, if val_every_n_steps is unset          | | |
|   |   |   - on_validation_end(metrics)                                       | | |
|   |   +----------------------------------------------------------------------+ | |
|   +----------------------------------------------------------------------------+ |
|   +----------------------------------------------------------------------------+ |
|   | FINAL VALIDATION   unless the last step was just validated                 | |
|   |   - on_validation_end(metrics)                                             | |
|   +----------------------------------------------------------------------------+ |
|   - on_fit_end(result)      (also on Ctrl-C or a crash, before re-raising)       |
+----------------------------------------------------------------------------------+
```

- An *accumulation boundary* is every `accumulate_grad_batches`-th batch, and the last
  batch of each epoch. With the default of 1, every batch is a boundary, so each batch
  fires all four step hooks.
- `on_after_backward` and `on_after_optimizer_step` are emitted by the helpers in
  `lwfm.training.steps`, so they fire only if `step_fn` calls them (`default_step` does).
- Validation only runs when `val_loader` and `metrics` are given; otherwise
  `on_validation_end` never fires.
- Within one hook, callbacks run in the order they were passed to `fit`.

In [5]:
from lwfm.training import Callback
from lwfm.training.callbacks import EarlyStopping


class PrintValidation(Callback):
    # Override only the hooks you need; the others do nothing
    def on_validation_end(self, state, metrics):
        top1 = metrics["top1"]
        print(f"epoch {state.epoch}, step {state.global_step}: top1 = {top1:.3f}")


seed_everything(0)
student = make_student()
graph = ExecutionGraph([student])
result = Trainer(device="cpu", max_epochs=50).fit(
    graph,
    train_loader,
    loss,
    torch.optim.SGD(student.parameters(), lr=0.1),
    val_loader=val_loader,
    metrics=metrics,
    # Stop once top1 has failed to improve for 2 validations in a row
    callbacks=[
        PrintValidation(),
        EarlyStopping("top1", max_validations_without_improvement=2),
    ],
)
result.stop_reason, result.epochs_completed  # stopped long before 50 epochs

epoch 1, step 20: top1 = 0.900
epoch 2, step 40: top1 = 0.887
epoch 3, step 60: top1 = 0.887


('stopped', 3)

## 4. Distillation: a frozen teacher is just another node

The teacher is a second node with `frozen=True`: it gets no gradients, and stays in
eval mode even though the trainer calls `graph.train()`. The distillation loss is one
more `LossBinding`, reading both nodes' logits. Nothing in the trainer changes.

Here the teacher is the model trained in section 3, and the student is a cheaper model
that only sees the image at half resolution. `LossWeightSchedule` ramps the KD term in,
so the student learns from labels first and from the teacher increasingly after.

In [6]:
import copy

import torch.nn.functional as F

from lwfm.training.callbacks import LossWeightSchedule


class KDLoss(nn.Module):
    """Hinton distillation: T^2 * KL(softmax(teacher/T) || softmax(student/T))."""

    def __init__(self, temperature: float = 4.0):
        super().__init__()
        self.t = temperature

    def forward(self, student_logits, teacher_logits):
        s = F.log_softmax(student_logits / self.t, dim=-1)
        t = F.log_softmax(teacher_logits / self.t, dim=-1)
        return F.kl_div(s, t, log_target=True, reduction="batchmean") * self.t**2


teacher = ModelNode(
    "teacher",
    copy.deepcopy(graph.deployable_model("student")),  # trained in section 3
    inputs="inputs.image",
    outputs="logits",
    frozen=True,
)
seed_everything(0)
small_student = ModelNode(
    "student",
    nn.Sequential(nn.AvgPool2d(2), nn.Flatten(), nn.Linear(3 * 4 * 4, K)),
    inputs="inputs.image",
    outputs="logits",
)
kd_graph = ExecutionGraph([small_student, teacher])
kd_loss = LossComposer(
    [
        LossBinding(
            "ce", nn.CrossEntropyLoss(), ["student.logits", "targets.class"], weight=0.5
        ),
        LossBinding("kd", KDLoss(), ["student.logits", "teacher.logits"], weight=0.5),
    ]
)
teacher_before = [p.clone() for p in teacher.parameters()]

result = Trainer(device="cpu", max_epochs=3, log_every_n_steps=20).fit(
    kd_graph,
    train_loader,
    kd_loss,
    # Only the student is optimised; the frozen teacher has nothing to train
    torch.optim.SGD(small_student.parameters(), lr=0.1),
    val_loader=val_loader,
    metrics=metrics,
    # The KD weight rises from 0 to 0.5 over the first 20 steps
    callbacks=[LossWeightSchedule("kd", lambda step: 0.5 * min(1.0, step / 20))],
    output_sink=ConsoleSink(),  # train/loss/ce and train/loss/kd are logged apart
)
unchanged = all(
    torch.equal(a, b) for a, b in zip(teacher_before, teacher.parameters(), strict=True)
)
print("teacher unchanged:", unchanged)

[step 20] train/loss/ce = 1.243


[step 20] train/loss/kd = 2.389


[step 20] train/loss/total = 1.756


[step 20] train/lr = 0.1


[step 20] train/epoch_seconds = 0.007203


[step 20] eval/val/top1 = 0.5


... (9 outputs left out)

[step 60] train/lr = 0.1


[step 60] train/epoch_seconds = 0.005261


[step 60] eval/val/top1 = 0.6125


teacher unchanged: True


## 5. Saving models through the sinks

Sinks record results, and never receive tensors. Saving a model is two steps: write
the file, then send an `artifact` event holding its path. `ModelCheckpoint` does both:

- `checkpoints/best.pt`: the student's weights, saved whenever `top1` improves;
- `checkpoints/last.pt`: the full training state, saved when training ends.

`LocalFileSink` writes the run directory. `OutputRouter` passes every event to each of
its sinks, and records the run's final status (completed, failed or interrupted).

In [7]:
import json
import tempfile

from lwfm.results.router import OutputRouter
from lwfm.results.sinks import LocalFileSink
from lwfm.training.callbacks import ModelCheckpoint

run_dir = Path(tempfile.mkdtemp()) / "demo_run"

seed_everything(0)
student = make_student()
graph = ExecutionGraph([student])
# fit runs inside the `with`, so a crash would be recorded as a failed run
with OutputRouter(LocalFileSink(run_dir)) as sink:
    Trainer(device="cpu", max_epochs=3).fit(
        graph,
        train_loader,
        loss,
        torch.optim.SGD(student.parameters(), lr=0.1),
        val_loader=val_loader,
        metrics=metrics,
        callbacks=[ModelCheckpoint(run_dir, node="student", monitored_metric="top1")],
        output_sink=sink,
    )

print(sorted(str(path.relative_to(run_dir)) for path in run_dir.rglob("*")))
# Artifact paths are relative to the run directory, so it can be moved or copied
for line in (run_dir / "events.jsonl").open():
    event = json.loads(line)
    if event["kind"] == "artifact":
        print(event["name"], "->", event["value"], "at step", event["step"])

['checkpoints', 'checkpoints/best.pt', 'checkpoints/last.pt', 'events.jsonl', 'manifest.json', 'status.json', 'summary.json']
checkpoint/best -> checkpoints/best.pt at step 20
checkpoint/last -> checkpoints/last.pt at step 60


`best.pt` holds weights only, not code: rebuild the same architecture, then load them into it. From here on it's a plain `nn.Module`.

In [8]:
from lwfm.evaluation.evaluator import Evaluator

restored = make_student()
weights = torch.load(run_dir / "checkpoints/best.pt", weights_only=True)
restored.module.load_state_dict(weights)
Evaluator().evaluate(
    ExecutionGraph([restored]), val_loader, metrics, device=torch.device("cpu")
)

EvaluationResult(metric_values={'top1': 0.8999999761581421})

## 6. Mistakes fail before training starts

`fit` checks its inputs before the first batch, so a mistake costs seconds rather than
a run: names that nothing produces, trainable parameters missing from the optimizer, a
scheduler without an interval, and misspelled callback hooks.

In [9]:
# A projector node the optimizer forgot: it would silently keep its random weights
student = make_student()
projector = ModelNode(
    "projector", nn.Linear(K, K), inputs="student.logits", outputs="z"
)
graph = ExecutionGraph([student, projector])
projected_loss = LossComposer(
    [LossBinding("ce", nn.CrossEntropyLoss(), ["projector.z", "targets.class"])]
)
try:
    Trainer(device="cpu", max_epochs=1).fit(
        graph,
        train_loader,
        projected_loss,
        torch.optim.SGD(student.parameters(), lr=0.1),  # projector left out
    )
except ValueError as error:
    print("ValueError:", error)

ValueError: trainable parameters missing from the optimizer: ['nodes.projector.module.weight', 'nodes.projector.module.bias']


In [10]:
class Typo(Callback):
    def on_validation_ended(self, state, metrics):  # should be on_validation_end
        print("never runs")


student = make_student()
try:
    Trainer(device="cpu", max_epochs=1).fit(
        ExecutionGraph([student]),
        train_loader,
        loss,
        torch.optim.SGD(student.parameters(), lr=0.1),
        callbacks=[Typo()],
    )
except ValueError as error:
    print("ValueError:", error)

ValueError: Typo defines ['on_validation_ended'], which are not hooks; the hooks are ['on_fit_start', 'on_epoch_start', 'on_step_start', 'on_after_backward', 'on_after_optimizer_step', 'on_step_end', 'on_epoch_end', 'on_validation_end', 'on_fit_end']


## 7. Loop settings, and many runs with one trainer

Mixed precision, gradient accumulation and gradient clipping are constructor arguments.
The trainer keeps no state between fits, so one instance can run a whole sweep: build
fresh objects for each run, call `fit`, and compare the results.

In [11]:
trainer = Trainer(
    device="cpu",
    max_epochs=3,
    precision="bf16",  # run the forward pass in bfloat16; weights stay fp32
    accumulate_grad_batches=2,  # one optimizer step per 2 batches (effective batch 64)
    gradient_clip_norm=1.0,  # clip the gradient norm before each optimizer step
)

best_top1 = {}
for lr in (0.01, 0.1, 0.5):
    seed_everything(0)  # the same initial weights for every learning rate
    student = make_student()
    result = trainer.fit(
        ExecutionGraph([student]),
        train_loader,
        loss,
        torch.optim.SGD(student.parameters(), lr=lr),
        val_loader=val_loader,
        metrics=metrics,
    )
    best_top1[lr] = round(result.best("top1")["top1"], 3)
best_top1

{0.01: 0.919, 0.1: 0.906, 0.5: 0.881}